In [91]:
import os
import sys
sys.path.insert(1,r'D:\Khabarov\Репозиторий\sql_premises_and_volumes')
sys.path.insert(1,r'D:\Khabarov\Репозиторий\sql_premises_and_volumes\Helpers')
from operator import index
from tokenize import group
import numpy as np
import pandas as pd
from Helpers.DbConnector import DbConnector
from Helpers.PremiseHelper import PremiseHelper
from Helpers.ParamsAndFuns import ParamsAndFuns as p
from Access.AccessInfo import AccessInfo as ai
from Helpers.MultiPremiseHelper import MultiPremiseHelper

source_path = r'D:\Khabarov\Репозиторий\sql_premises_and_volumes\SourceData\ИсходныеДанные_АнализОтделки_ПД.xlsx'

dbCon = DbConnector()
multi_prems = MultiPremiseHelper(source_path,dbCon)

Подключено
Добавлен АКД06 Стадия П
Добавлен ЛБД01.2 Стадия П
Добавлен ЕСН1.01.1 Стадия П
Добавлен ГРД01 Стадия П
Добавлен ЕБГ32 Стадия П
Добавлен ЛТН01.1 Стадия П
Добавлен КВС01 Стадия П
Добавлен КБВ01 Стадия П
Добавлен ТКП01 Стадия П
Добавлен ТКП02 Стадия П
Добавлен ЗРК02.1 Стадия П
Добавлен РПТ06.2 Стадия П
Добавлен ЮКВ08 Стадия П
Добавлен ЮКВ05 Стадия П
Добавлен РСП05.1 Стадия П
Добавлен РСП05.2 Стадия П


In [ ]:
directory = r'D:\Khabarov\Репозиторий\sql_premises_and_volumes\Data'
multi_prems.save_etp_form(directory=directory)

#Для выгрузки ЕТП

In [ ]:
multi_prems.show_new_premises_for_dictionary()

#Посмотреть какие помещения добавить

In [ ]:
multi_prems.add_revit_ids_to_premises()

#Добавить к датафрейму айдишники ревита

In [ ]:
multi_prems.get_floor_types_features_by_premises()

#Выгрузить признаки для определения ML типа этажа

In [ ]:
multi_prems.get_unique_attr_of_flast()

#Выгрузить квартиры с характеристикой уникальности

In [ ]:
multi_prems.get_vehicles_in_living()

#Выгрузить велосипедные и колясочные

### Площади квартир для анализа отделки

In [92]:
from datetime import datetime

df = multi_prems.dfFull

#Берем помещения квартир
premises = df[df[p.bru_destination_pn] == 'Жилье']
premises = premises.copy()[[
            'Наименование ОС'
            ,p.adsk_premise_number
            ,p.section_str_pn
            ,p.rooms_count
            ,p.rooms_sale_count
            ,p.bru_floor_int_pn
            ,p.name_pn
            ,p.bru_premise_part_area_pn
            ,"Антресоль"
            ,"Дуплекс"
            ,"С цокольным этажом"
            ,"Терраса на кровле"
            ,"Терраса на земле"
            ,"Летняя кухня на крыше"
            ,"Второй свет"
            ,"Отдельный вход"
            ,"Пентхаус"
            ,"Свободная планировка"
            ,"Стадия"
            ,"Высота потолка от пола"
            ,p.bru_premise_full_area_pn
            ,p.bru_premise_non_summer_area_pn
            ,p.bru_premise_summer_area_pn
            ,p.adsk_type_pn
            ,p.bru_type_pn
            ]]

#Определяем уникальность
premises["Уникальность"] = (premises["Терраса на кровле"] 
                        + premises["Терраса на земле"]
                        + premises["Антресоль"]
                        + premises["Дуплекс"]
                        + premises["С цокольным этажом"]
                        + premises["Летняя кухня на крыше"]
                        + premises["Второй свет"]
                        + premises["Отдельный вход"]
                        + premises["Пентхаус"]
                        + premises["Свободная планировка"]
                        ) > 0


res = premises.groupby(['Наименование ОС','Стадия',p.adsk_premise_number],as_index=False).agg(
    free_plan=("Свободная планировка","all"),
    unique=("Уникальность","all"),
    s_full=(p.bru_premise_full_area_pn,"max"),
    s_without_summer=(p.bru_premise_non_summer_area_pn,"max"),
    s_summer=(p.bru_premise_summer_area_pn,"max"),
    proj_type=(p.adsk_type_pn,"first"),
    sales_type=(p.bru_type_pn,'first'),
    rooms_count=(p.rooms_count,'max'),
    sales_rooms_count=(p.rooms_sale_count,'max'),
    prems_count = (p.adsk_premise_number,'count'),
    height_max = ("Высота потолка от пола",'max'),
    height_min = ("Высота потолка от пола",'min'),
    height_median = ("Высота потолка от пола",'median'),
    compound=(p.name_pn, lambda x: sorted(list(zip(x, premises.loc[x.index, p.bru_premise_part_area_pn]))))
).sort_values(
    ["Наименование ОС","Стадия","Номер квартиры"],ascending=True
)

res = res.rename(
    {"Наименование ОС":"сo_name","Стадия":"co_stage",p.adsk_premise_number:"flat_num"}
    ,axis=1)




directory = r'D:\Khabarov\Репозиторий\sql_premises_and_volumes\Data'

name = "apartment_finishing"
stages_dict = {
    "Архпрограмма":"archprogram-stage",
    "Концепция планировок": "concept-stage",
    "Стадия П": "documentation-stage",
    "Стадия РД": "dev-stage"
}
stage_ru = res['co_stage'].to_list()[0]
stage = stages_dict.get(stage_ru)
today = datetime.now().strftime("%m-%Y") 

full_path = directory + f"\{name}_{stage}_{today}.csv"
res.to_csv(full_path,sep=';',encoding='UTF-8',index=False)